# Week 8: single-nucleus ATACseq using SnapATAC2

## 0. Introduction

In RNA sequencing experiments, we take a snapshot of a cell's current expression profile by sequencing the mRNA captured by millions of poly-dT primers (against the poly-A tail). However, RNA represents the final output of a highly regulated process, not the regulatory logic that produced it. Much of this logic is encoded at the level of chromatin structure. In order to be expressed, tightly compacted chromatin must first be made physically accessible through epigenetic modifications such as histone post-translational modification, DNA methylation, and ATP-dependent chromatin remodeling. Accessible regions of DNA **permit** transcription factor binding at regulatory elements such as promoters and enhancers, and these interactions ultimately lead to the gene expression programs which define a cell's identity.

**ATAC-seq (Assay for Transposase Accessible Chromatin using sequencing)** measures which regions of the genome are physically accessible inside each cell. The method uses a transposase enzyme (Tn5) that makes cuts in accessible chromatin regions and simultaneously inserts sequencing adapters, yielding many short fragments whose ends are compatible with sequencing. Each fragment therefore represents an snippet from an accessible location in the genome, and regions which accumulate sequencing reads generally correspond to active regulatory elements such as promoters and enhancers. Because chromatin accessibility patterns are generally more stable and often precede detectable transcriptional changes, ATAC-seq can be a powerful tool for identifying cell states, lineage commitment, and early regulatory responses before they are observable by RNA expression alone.

<div style="display: flex; justify-content: center;">
    <img src="images/Active ATAC-Seq Kit workflow.png" width="600">
</div>

In today's class we will analyze single-cell ATACseq data from peripheral blood mononuclear cells (PBMCs), a mixture of immune cell types including T cells, B cells, NK cells, and monocytes. PBMCs are a useful test case because many immune populations share similar transcriptional profiles at rest. For example, naive and memory T cells can appear nearly identical using RNA-seq alone. However, their regulatory architecture is already different: each subtype opens distinct transcription factor binding sites that prepare it to respond to specific stimuli.

Our goal today is to cluster and annotate PBMC populations based on their chromatin accessibility profiles. We will identify groups of cells that share similar accessibility landscapes and assign cell identities based on characteristic regulatory features. We will then improve these annotations by integrating RNA-seq data, allowing expression information to refine and validate cell type labels and illustrating how epigenomic and transcriptional measurements together define cellular identity.

Before we begin, make sure snapATAC2 is installed in a new environment by following the [README](README.md).

# Part I: Analyzing the 5K PBMC dataset
## 1. Import library and fragment file
In a typical single-cell ATAC-seq experiment, hundreds of millions to billions of DNA fragments are sequenced. However, each individual cell contributes only a few thousand fragments sampled from a genome containing hundreds of thousands of regulatory regions. As a result, most regions appear absent in any one cell simply because they were not sampled, not because the chromatin is truly closed. This extreme sparsity makes it difficult to directly determine which regulatory elements are active in each cell.

SnapATAC2, developed in the laboratory of Bing Ren, addresses this by first comparing cells based on broad patterns across the entire genome rather than trying to identify regulatory regions one cell at a time. It emphasizes genomic regions that are informative for distinguishing cell types while down-weighting regions that appear in nearly every cell. The method then groups together cells that share similar accessibility patterns. After cells are grouped, their signals are combined to reliably detect regulatory elements and infer the transcription factors likely controlling each population. In this way, very sparse measurements from individual cells are integrated into coherent and biologically interpretable chromatin states.

Let's begin by defining our pre-loaded fragment file.

In [ ]:
import snapatac2 as snap
import warnings
warnings.filterwarnings("ignore")
snap.__version__

In [ ]:
fragment_file = snap.datasets.pbmc5k()
fragment_file

## 2. Preprocessing
Import the fragments file (this can take awhile).

In [ ]:
data = snap.pp.import_fragments(
    fragment_file,
    chrom_sizes=snap.genome.hg38,
    file="pbmc.h5ad",
    sorted_by_barcode=False,
)
data

Next, let's look at the distribution of fragment sizes:

In [ ]:
snap.pl.frag_size_distr(data, interactive=True)

We see three peaks. The first peak consists of short fragments, corresponding to the fragments of open chromatin. The second peak (200bp) is produced due to Tn5 transposase making cuts at linker DNA regions between nucleosomes. Each nucleosome is wrapped by ~150bp, with ~50bp linker DNA between nucleosomes, leading to periodic peaks every 200bp. The third peak (400bp), therefore, represents where a cut was made on either side of 2 adjacent nucleosomes.

In [ ]:
fig = snap.pl.frag_size_distr(data, show=False)
fig.update_yaxes(type="log")
fig.show()

Active regulatory regions are enriched around transcription start sites (TSS). This computes a per-cell quality score measuring how strongly reads pile up at promoters.\
High TSS enrichment = intact nuclei and good chromatin signal\
Low TSS enrichment = debris or background DNA

In [ ]:
snap.metrics.tsse(data, snap.genome.hg38)

In [ ]:
snap.pl.tsse(data, interactive=True)

## Exercise 1
Looking at the cell fragment counts and TSSE scores, a cluster of cells appears to be higher quality than the rest. Fill in the empty arguments below to filter in only the high quality cells. \
\
*For additional arguments and explanations, see the SnapATAC2 [API reference](https://scverse.org/SnapATAC2/api/index.html).*

In [ ]:
#your answer here
snap.pp.filter_cells(data, min_counts=, min_tsse=, max_counts=)
data

<details>
<summary><b>Show answer</b></summary>

```python
    snap.pp.filter_cells(data, min_counts=5000, min_tsse=10, max_counts=100000)

In [ ]:
snap.pp.add_tile_matrix(data)

In [ ]:
snap.pp.select_features(data, n_features=250000)

## 3. Doublet removal

In droplet single-cell experiments, sometimes two cells enter the same droplet and receive one barcode. The sequencer thinks it is one cell, but its chromatin accessibility looks like a mixture of two cell types. These are called doublets.

If not removed, they create fake intermediate cell states and distort clustering and downstream biology.

In [ ]:
snap.pp.scrublet(data)

In [ ]:
snap.pp.filter_doublets(data)
data

## 4. Dimensionality Reduction and Clustering
To visualize our data in two dimensions, we must create a low-dimensional representation and a nearest neighbor graph.

In [ ]:
snap.tl.spectral(data)

In [ ]:
snap.pp.knn(data)
snap.tl.leiden(data)

Now we can compute and plot the UMAP.

In [ ]:
snap.tl.umap(data)

In [ ]:
snap.pl.umap(data, color='leiden', interactive=True, height=500)

## 5. Cell cluster annotation
Now that we have clusters, we will attempt to annotate clusters based on gene activity scores for each cell.

In [ ]:
gene_matrix = snap.pp.make_gene_matrix(data, snap.genome.hg38)
gene_matrix

In [ ]:
import scanpy as sc

sc.pp.filter_genes(gene_matrix, min_cells= 5)
sc.pp.normalize_total(gene_matrix)
sc.pp.log1p(gene_matrix)

In [ ]:
sc.external.pp.magic(gene_matrix, solver="approximate")

In [ ]:
# Copy over UMAP embedding
gene_matrix.obsm["X_umap"] = data.obsm["X_umap"]

In [ ]:
marker_genes = ['MS4A1', 'CD3D', 'LEF1', 'NKG7', 'TREM1', 'LYZ', 'PPBP']
sc.pl.umap(gene_matrix, use_raw=False, color=["leiden"] + marker_genes)

In [ ]:
import pandas as pd

marker_sets = {
    # T cells
    "CD4 memory": ["S100A4","ANXA1","KLF2","GPR183","TNFRSF4"],
    "CD8 T":  ["CD8A", "CD8B", "GZMK", "CCL5", "NKG7"],
    "Treg":   ["FOXP3", "IL2RA", "CTLA4", "IKZF2"],
    
    # B cells
    "B cell (naive)":    ["MS4A1", "CD79A", "CD74", "HLA-DRA", "TCL1A"],
    "B cell (memory)":   ["MS4A1", "CD79A", "CD74", "HLA-DRA", "CD27"],
    "Plasma": ["MZB1", "JCHAIN", "XBP1", "SDC1"],
   
    # Monocytes
    "CD14 Mono": ["LYZ", "S100A8", "S100A9", "FCN1", "LGALS3", "CTSS"],
    "CD16 Mono": ["LYZ", "FCGR3A", "MS4A7", "LST1", "IFITM3"],

    # Dendritic cells
    "Dendritic": ["FCER1A", "CST3", "CD1C", "CLEC10A", "IL3RA", "TCF4"],

    # Platelets
    "Platelet": ["PPBP", "PF4", "ITGA2B", "GP9", "SDPR"],
}

# Keep only markers present in your data
present = set(gene_matrix.var_names)
marker_sets = {ct: [g for g in gs if g in present] for ct, gs in marker_sets.items()}
marker_sets = {ct: gs for ct, gs in marker_sets.items() if len(gs) >= 2}  # require >=2 markers

# Score each cell for each marker set
for ct, genes in marker_sets.items():
    sc.tl.score_genes(gene_matrix, gene_list=genes, score_name=f"score_{ct}", use_raw=False)

score_cols = [f"score_{ct}" for ct in marker_sets.keys()]

# Average scores per cluster
cluster_scores = (
    gene_matrix.obs[["leiden"] + score_cols]
    .groupby("leiden")
    .mean()
)

# Assign each Leiden cluster the top-scoring cell type
best = cluster_scores.idxmax(axis=1).str.replace("score_", "", regex=False)

gene_matrix.obs["cell_type"] = gene_matrix.obs["leiden"].map(best).astype("category")

# Plot UMAP
sc.pl.umap(
    gene_matrix,
    color="cell_type",
    legend_loc="right margin",
    title="UMAP by cell type"
)

In [ ]:
gene_matrix.write("pbmc5k_gene_mat.h5ad", compression='gzip')

## Exercise 2
Looking back to our marker gene plots, it looks like we should be able to define an additional cell type cluster where "" is highly accessible. Find which cell this marker gene corresponds to and create a new marker set to define these cells. Then, plot the UMAP with the additional cell type added.

In [ ]:
#your answer here

<details>
<summary><b>Show answer</b></summary>

```python
marker_sets = {
    # T cells
    "CD4 naive":  ["CCR7","IL7R","TCF7","LEF1","SELL","LTB","MAL"],
    "CD4 memory": ["S100A4","ANXA1","KLF2","GPR183","TNFRSF4"],
    "CD8 T":  ["CD8A", "CD8B", "GZMK", "CCL5", "NKG7"],
    "Treg":   ["FOXP3", "IL2RA", "CTLA4", "IKZF2"],
    
    # B cells
    "B cell (naive)":    ["MS4A1", "CD79A", "CD74", "HLA-DRA", "TCL1A"],
    "B cell (memory)":   ["MS4A1", "CD79A", "CD74", "HLA-DRA", "CD27"],
    "Plasma": ["MZB1", "JCHAIN", "XBP1", "SDC1"],
   
    # NK
    "NK":     ["NKG7", "GNLY", "PRF1", "KLRD1", "FCGR3A"],

    # Monocytes
    "CD14 Mono": ["LYZ", "S100A8", "S100A9", "FCN1", "LGALS3", "CTSS"],
    "CD16 Mono": ["LYZ", "FCGR3A", "MS4A7", "LST1", "IFITM3"],

    # Dendritic cells
    "Dendritic": ["FCER1A", "CST3", "CD1C", "CLEC10A", "IL3RA", "TCF4"],

    # Platelets
    "Platelet": ["PPBP", "PF4", "ITGA2B", "GP9", "SDPR"],
}

present = set(gene_matrix.var_names)
marker_sets = {ct: [g for g in gs if g in present] for ct, gs in marker_sets.items()}
marker_sets = {ct: gs for ct, gs in marker_sets.items() if len(gs) >= 2}  # require >=2 markers

for ct, genes in marker_sets.items():
    sc.tl.score_genes(gene_matrix, gene_list=genes, score_name=f"score_{ct}", use_raw=False)

score_cols = [f"score_{ct}" for ct in marker_sets.keys()]

cluster_scores = (
    gene_matrix.obs[["leiden"] + score_cols]
    .groupby("leiden")
    .mean()
)

best = cluster_scores.idxmax(axis=1).str.replace("score_", "", regex=False)

gene_matrix.obs["cell_type"] = gene_matrix.obs["leiden"].map(best).astype("category")

sc.pl.umap(
    gene_matrix,
    color="cell_type",
    legend_loc="right margin",
    title="UMAP by cell type"
)

# Part II: Annotating cell clusters by integrating single-cell RNA-seq data
Now that we've completed an initial clustering and annotation with our ATAC-seq data, we are going to use another example to show how RNA-seq data can be integrated for more refined cell-type annotation. Single-cell analysis allows us to better understand cellular heterogeneity in tissue, but a crucial step before we get there is annotating cell types. Currently, most analyses use RNA-sequencing data to classify subtypes. As techniques and scientific understanding continue to develop, multiomic approaches are becoming instrumental in forming a broader understanding of "what is a cell type".

Single-cell ATAC-seq offers information about accessible chromatin regions across the genome, while RNA-seq offers information about gene expression. Multiomic approaches allow for extraction of both types of information to gain a more holistic understanding of cell types, particularly in capturing cell states that cannot be found at the RNA level. For this exercise, we are going to integrate these data by training a model on a reference RNA-seq dataset to better classify cell types in our ATAC-seq dataset.

## 1. Import sequencing datasets

In [ ]:
import snapatac2 as snap
import scanpy as sc
import pandas as pd
import anndata as ad
import scvi
import numpy as np
import warnings
warnings.filterwarnings("ignore")

scvi.settings.seed = 0
snap.__version__

First, we will import the reference single-cell RNA-seq dataset with annotated cells. 

In [ ]:
reference = snap.read(snap.datasets.pbmc10k_multiome(), backed=None)
reference

Next, we will import the cell-by-bin matrix of single-cell ATAC-seq data. This is our query data.

In [ ]:
atac = snap.read(snap.datasets.pbmc5k(type='h5ad'), backed=None)
query = snap.pp.make_gene_matrix(atac, gene_anno=snap.genome.hg38)
query

Finally, we will merge the reference and query data together and use the scanpy library to find highly variable genes. 

In [ ]:
query.obs['cell_type'] = pd.NA
data = ad.concat(
    [reference, query],
    join='inner',
    label='batch',
    keys=["reference", "query"],
    index_unique='_',
)
data

In [ ]:
sc.pp.filter_genes(data, min_cells=5)
sc.pp.highly_variable_genes(
    data,
    n_top_genes = 5000,
    flavor="seurat_v3",
    batch_key="batch",
    subset=True
)

## 2. Data Integration and Model Training

First, we setup the scvi-tools to pretrain the model.

In [ ]:
scvi.model.SCVI.setup_anndata(data, batch_key="batch")
vae = scvi.model.SCVI(
    data,
    n_layers=2,
    n_latent=30,
    gene_likelihood="nb",
    dispersion="gene-batch",
)

Now, we pretrain!

In [ ]:
vae.train(max_epochs=1000, early_stopping=True)

## Exercise 3
The above cell might might take awhile to compute... How might we speed this up for now?

In [ ]:
#your answer here

<details>
<summary><b>Show answer</b></summary>

```python
vae.train(max_epochs=10, early_stopping=True)

We can also plot the training history to make sure the model has converged. After training, scvi-tools stores training metrics in a dictionary called history.

In [ ]:
ax = vae.history['elbo_train'][1:].plot()
vae.history['elbo_validation'].plot(ax=ax)

Here is what it would look like if you were to train with 1000 epochs.

<img src="images/PretrainHistory_1000epochs.png" width="600">

Here, we are creating labels for our semi-supervised cell-type annotation.

In [ ]:
data.obs["celltype_scanvi"] = 'Unknown'
ref_idx = data.obs['batch'] == "reference"
data.obs["celltype_scanvi"][ref_idx] = data.obs['cell_type'][ref_idx]

Next, we are creating a semi-supervised cell type classifier from the trained SCVI model using scvi-tools.

In [ ]:
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    adata=data,
    labels_key="celltype_scanvi",
    unlabeled_category="Unknown",
)

Now, we can train the model while setting a couple important parameters. The number of epochs refers to how many times the model will run through the data. The number of samples per label refers to how many cells of each label (or cell type) the model will sample per epoch. This is important to specify, as some cell types are much more numerous than others.

In [ ]:
lvae.train(max_epochs=10, n_samples_per_label=100)

Again, we can view the training history. 

In [ ]:
lvae.history['elbo_train'][1:].plot()

And again, here is what this would look like with 1000 epochs.

<img src="images/FullTrain_1000epochs.png" width="600">

Now, we will perform the label transfer/prediction and join the reference and query data. We will extract the final results and store them inside our AnnData object.

In [ ]:
data.obs["C_scANVI"] = lvae.predict(data)
data.obsm["X_scANVI"] = lvae.get_latent_representation(data)

## 3. Visualization of Integrated Datasets

In [ ]:
sc.pp.neighbors(data, use_rep="X_scANVI")
sc.tl.umap(data)

In [ ]:
sc.pl.umap(data, color=['C_scANVI', "batch"], wspace=0.45)

Here is with 1000 epochs.

<img src="images/firstUMAP.png" width="1000">

Save the predicted cell type labels back to the original cell by bin matrix.

In [ ]:
atac.obs['cell_type'] = data.obs.loc[atac.obs_names + '_query']['C_scANVI'].to_numpy()

## 4. Evaluate Training Performance on Annotation

Reflect: how well do the leiden clusters match the cell type annotation? 

Note: the ATAC-seq data has fewer cells and therefore does not have the power to differentiate all cell types in the RNA-seq annotated data.

In [ ]:
sc.pl.umap(atac, color=['cell_type', "leiden"], wspace=0.45)

And with 1000 epochs.

<img src="images/secondUMAP.png" width="1000">

Save annotated data for future use.

In [ ]:
atac.write("pbmc5k_annotated.h5ad", compression="gzip")

## Bonus Exercise
Congrats on making it through the analysis!
Now, try going through the analysis using data from one of the loaded datasets.